# GeoSustain: Random Forest for seven crops

One classifier: **Random Forest**. Crop classes: Banana, Cacao, Coconut, Durian, Mango, Rice and Rubber. Seven inputs: temperature, humidity, soil pH, rainfall, NDVI, elevation and slope. **NPK excluded.**

**Chapter 2 default:** results remain **N/A**. Training is disabled until you set `RUN_TRAINING = True`. This starter contains no executed outputs or previously computed performance scores. The existing-artifact reference is kept separate from any new run.


## 1. Prepare the runtime

A CPU runtime is sufficient. The source CSV and Python code are embedded below.


In [ ]:
import sys, subprocess, json, base64
from pathlib import Path
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DEPENDENCIES = ['scikit-learn==1.8.0', 'numpy>=1.26,<3', 'pandas>=2.2,<4', 'joblib>=1.4,<2', 'matplotlib>=3.8,<4', 'jinja2>=3.1,<4']
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *DEPENDENCIES])
PLATFORM = "google_colab" if IN_COLAB else "local"


## 2. Set the experiment

Keep training disabled for the Chapter 2 N/A table. Only enable it when you intend to conduct and report an actual run. No algorithm selection or comparison is performed.


In [ ]:
RUN_TRAINING = False
RESEARCHER_NAME = ""
SEED = 42
CV_FOLDS = 5
FOREST_TREES = 500
WORK = Path("/content/geosustain_rf_seven") if IN_COLAB else Path("geosustain_rf_seven")
WORK.mkdir(parents=True, exist_ok=True)


## 3. Restore the exact dataset and training code

The CSV matches the dataset hash recorded for the installed seven-input Random Forest. The reference JSON identifies that existing artifact; a new run is recorded separately.


In [ ]:
DATASET_BASE64 = '...'
TRAINER_SOURCE = '"""Random Forest only, seven crop classes, no NPK. Research outputs only."""\nfrom datetime import datetime, timezone\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport platform\nfrom time import perf_counter\nimport uuid\nimport zipfile\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix\nfrom sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split\n\nFEATURES = [\'temperature\', \'humidity\', \'ph\', \'rainfall\', \'ndvi\', \'elevation\', \'slope\']\nLABELS = [\'banana\', \'cacao\', \'coconut\', \'durian\', \'mango\', \'rice\', \'rubber\']\n\n\ndef file_hash(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n\n\ndef prepare_data(dataset, manifest_path):\n    raw = Path(dataset).read_bytes()\n    manifest = json.loads(Path(manifest_path).read_text(encoding=\'utf-8\'))\n    if manifest.get(\'dataset_sha256\') != hashlib.sha256(raw).hexdigest():\n        raise ValueError(\'Dataset bytes differ from the bundled provenance manifest. Review and document a new dataset before training.\')\n    if manifest.get(\'feature_order\') != FEATURES or manifest.get(\'classes\') != LABELS:\n        raise ValueError(\'Expected the documented seven-input, seven-crop, no-NPK experiment.\')\n    frame = pd.read_csv(io.BytesIO(raw), dtype={\'_record_id\': str, \'_source_basis\': str})\n    if not set(FEATURES + [\'label\', \'_record_id\', \'_source_basis\']) <= set(frame):\n        raise ValueError(\'CSV is missing documented predictors, label or source identifiers.\')\n    if {\'N\', \'P\', \'K\'} & set(frame):\n        raise ValueError(\'NPK columns are excluded from this experiment, not replaced with zeros or guessed values.\')\n    if not 70 <= len(frame) <= 100000:\n        raise ValueError(\'Expected 70–100,000 source rows.\')\n    for key in [\'label\', \'_record_id\', \'_source_basis\']:\n        if frame[key].isna().any() or frame[key].astype(str).str.strip().eq(\'\').any():\n            raise ValueError(f\'Missing {key}; source rows are not silently filled.\')\n    if frame[\'_record_id\'].duplicated().any():\n        raise ValueError(\'Source record identifiers must be unique.\')\n    frame[\'label\'] = frame.label.astype(str).str.strip().str.lower()\n    if set(frame.label) != set(LABELS):\n        raise ValueError(\'Existing labelled rows must cover exactly the seven approved crops.\')\n    if any(pd.api.types.is_bool_dtype(frame[key]) for key in FEATURES):\n        raise ValueError(\'Boolean predictors are not numeric measurements.\')\n    frame[FEATURES] = frame[FEATURES].apply(pd.to_numeric, errors=\'raise\')\n    if not np.isfinite(frame[FEATURES].to_numpy(dtype=float)).all():\n        raise ValueError(\'Missing/nonfinite values need source review; no imputation is performed.\')\n    if frame.groupby(FEATURES, dropna=False).label.nunique().gt(1).any():\n        raise ValueError(\'Identical predictors have conflicting crop labels; resolve those records before training.\')\n    source_count = len(frame)\n    frame = frame.drop_duplicates(FEATURES + [\'label\']).reset_index(drop=True)\n    if frame.label.value_counts().min() < 10:\n        raise ValueError(\'Every crop needs at least ten distinct labelled input rows.\')\n    audit = {\'dataset_sha256\': hashlib.sha256(raw).hexdigest(), \'input_rows\': source_count,\n        \'duplicate_rows_removed_before_split\': source_count - len(frame), \'unique_rows\': len(frame),\n        \'class_counts\': {k: int(v) for k, v in frame.label.value_counts().sort_index().items()},\n        \'source_basis_counts\': {k: int(v) for k, v in frame[\'_source_basis\'].value_counts().items()},\n        \'rows_generated_this_run\': 0, \'period_conversion\': \'none\', \'npk_used\': False,\n        \'independent_farm_groups\': \'unknown; row-level experiment only\'}\n    return frame, manifest, audit\n\n\ndef make_partitions(frame, seed=42, cv_folds=5):\n    if type(seed) is not int or not 0 <= seed < 2**32:\n        raise ValueError(\'Seed must be an unsigned 32-bit integer.\')\n    if type(cv_folds) is not int or not 2 <= cv_folds <= 10:\n        raise ValueError(\'Cross-validation folds must be 2–10.\')\n    train, holdout = train_test_split(np.arange(len(frame)), test_size=.2,\n        stratify=frame.label, random_state=seed)\n    if frame.iloc[train].label.value_counts().min() < cv_folds:\n        raise ValueError(\'Not enough training rows per crop for the requested CV folds.\')\n    keys = pd.util.hash_pandas_object(frame[FEATURES], index=False).astype(str)\n    assert not set(keys.iloc[train]) & set(keys.iloc[holdout])\n    cv = list(StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed + 1)\n              .split(frame.iloc[train][FEATURES], frame.iloc[train].label))\n    assignments = {}\n    for fold, (a, b) in enumerate(cv):\n        assert not set(keys.iloc[train[a]]) & set(keys.iloc[train[b]])\n        for j in train[b]: assignments[int(j)] = fold\n    split = frame[[\'_record_id\', \'label\', \'_source_basis\']].copy()\n    split[\'partition\'] = [\'train\' if i in set(train.tolist()) else \'holdout\' for i in range(len(frame))]\n    split[\'cv_validation_fold\'] = [assignments.get(i) for i in range(len(frame))]\n    split[\'input_fingerprint\'] = keys\n    return train, holdout, cv, split\n\n\ndef run_training(dataset, manifest_path, reference_path, output_parent, *, seed=42,\n                 trees=500, cv_folds=5, execution_platform=\'local\', researcher=\'Not supplied\'):\n    """Fit one predefined RF on training rows; score an untouched test partition."""\n    if type(trees) is not int or not 1 <= trees <= 1000:\n        raise ValueError(\'Tree count must be 1-1000.\')\n    frame, manifest, audit = prepare_data(dataset, manifest_path)\n    reference = json.loads(Path(reference_path).read_text(encoding=\'utf-8\'))\n    if reference[\'dataset_sha256\'] != audit[\'dataset_sha256\'] or reference[\'feature_order\'] != FEATURES:\n        raise ValueError(\'Dataset/input contract differs from the baseline reference.\')\n    train, holdout, cv, split = make_partitions(frame, seed, cv_folds)\n    started = datetime.now(timezone.utc)\n    run_id = started.strftime(\'%Y%m%dT%H%M%SZ\') + \'-\' + uuid.uuid4().hex[:8]\n    output = Path(output_parent) / run_id\n    if any(part.lower() == \'backend\' for part in output.resolve().parts):\n        raise ValueError(\'Research outputs cannot be written into backend/.\')\n    output.mkdir(parents=True, exist_ok=False)\n    x_train, y_train = frame.iloc[train][FEATURES], frame.iloc[train].label\n    x_test, y_test = frame.iloc[holdout][FEATURES], frame.iloc[holdout].label\n    model = RandomForestClassifier(n_estimators=trees, max_depth=14, min_samples_leaf=6,\n        min_samples_split=12, max_features=\'sqrt\', class_weight=\'balanced_subsample\',\n        random_state=seed, n_jobs=1)\n    scoring = {key: key for key in [\'accuracy\', \'precision_macro\', \'recall_macro\', \'f1_macro\']}\n    print(f\'Random Forest: {cv_folds} training-only cross-validation folds\', flush=True)\n    scores = cross_validate(model, x_train, y_train, cv=cv, scoring=scoring,\n                            n_jobs=1, error_score=\'raise\', return_train_score=False)\n    cv_rows = [{\'fold\': fold + 1, **{key: float(scores[\'test_\' + key][fold]) for key in scoring}}\n               for fold in range(cv_folds)]\n    tick = perf_counter()\n    model.fit(x_train, y_train)\n    fit_seconds = perf_counter() - tick\n    prediction = model.predict(x_test)\n    report = classification_report(y_test, prediction, labels=LABELS, output_dict=True, zero_division=0)\n    matrix = confusion_matrix(y_test, prediction, labels=LABELS)\n    metrics = {\'algorithm\': \'Random Forest\', \'holdout_accuracy\': float(accuracy_score(y_test, prediction)),\n        \'holdout_precision_macro\': float(report[\'macro avg\'][\'precision\']),\n        \'holdout_recall_macro\': float(report[\'macro avg\'][\'recall\']),\n        \'holdout_f1_macro\': float(report[\'macro avg\'][\'f1-score\']),\n        \'holdout_f1_weighted\': float(report[\'weighted avg\'][\'f1-score\']),\n        **{f\'cv_{key}_mean\': float(scores[\'test_\' + key].mean()) for key in scoring}}\n    pd.DataFrame([metrics]).to_csv(output / \'metrics.csv\', index=False)\n    pd.DataFrame(report).T.to_csv(output / \'classification_report.csv\')\n    pd.DataFrame(matrix, index=LABELS, columns=LABELS).to_csv(output / \'confusion_matrix.csv\', index_label=\'actual_crop\')\n    pd.DataFrame(cv_rows).to_csv(output / \'cross_validation.csv\', index=False)\n    split.to_csv(output / \'split_manifest.csv\', index=False)\n    frame.to_csv(output / \'training_rows_deduplicated.csv\', index=False)\n    predictions = frame.iloc[holdout][[\'_record_id\', \'label\', \'_source_basis\']].copy()\n    predictions[\'predicted_crop\'] = prediction\n    for index, crop in enumerate(model.classes_):\n        predictions[\'probability_\' + crop] = model.predict_proba(x_test)[:, index]\n    predictions.to_csv(output / \'holdout_predictions.csv\', index=False)\n    joblib.dump(model, output / \'random_forest.joblib\')\n    restored = joblib.load(output / \'random_forest.joblib\')\n    np.testing.assert_array_equal(restored.predict(x_test), prediction)\n    np.testing.assert_allclose(restored.predict_proba(x_test), model.predict_proba(x_test), rtol=0, atol=0)\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import ConfusionMatrixDisplay\n    fig, ax = plt.subplots(figsize=(8, 7))\n    ConfusionMatrixDisplay(matrix, display_labels=LABELS).plot(ax=ax, xticks_rotation=45, cmap=\'Greens\', colorbar=False)\n    ax.set_title(\'Random Forest: held-out dataset rows\')\n    fig.tight_layout(); fig.savefig(output / \'confusion_matrix.png\', dpi=160); plt.close(fig)\n    pd.DataFrame({\'feature\': FEATURES, \'global_impurity_importance\': model.feature_importances_}).to_csv(\n        output / \'feature_importance.csv\', index=False)\n    record = {\'run_id\': run_id, \'algorithm\': \'Random Forest\',\n        \'started_at_utc\': started.isoformat(), \'finished_at_utc\': datetime.now(timezone.utc).isoformat(),\n        \'researcher\': researcher, \'execution_platform_reported\': execution_platform,\n        \'environment\': {\'python\': platform.python_version(), \'sklearn\': sklearn.__version__,\n                        \'numpy\': np.__version__, \'pandas\': pd.__version__, \'joblib\': joblib.__version__},\n        \'features\': FEATURES, \'classes\': LABELS, \'hyperparameters\': model.get_params(),\n        \'split\': {\'seed\': seed, \'train_rows\': len(train), \'holdout_rows\': len(holdout),\n                  \'cv_folds\': cv_folds, \'exact_predictor_overlap\': 0, \'test_used_for_selection\': False},\n        \'algorithm_selection\': \'Random Forest specified before evaluation; no algorithm comparison.\',\n        \'data_audit\': audit, \'metrics\': metrics, \'fit_seconds\': fit_seconds,\n        \'training_script_sha256\': file_hash(__file__),\n        \'new_artifact_sha256\': file_hash(output / \'random_forest.joblib\'),\n        \'baseline_reference\': reference, \'production_model_changed\': False,\n        \'fit_scope\': \'training partition only; exported model replays held-out predictions\',\n        \'npk_used\': False, \'field_validation\': \'not established\',\n        \'annual_input_compatibility\': \'not established\',\n        \'limitations\': [\'Existing mixed Kaggle-based and generated profile/GIS rows; not independent farm accuracy.\',\n          \'Independent farm groups and consistent labelled climate periods are unavailable.\',\n          \'Banana variety labels and NPK measurements are not available in these rows.\',\n          \'Recorded platform/timestamps support replay but do not independently authenticate the place of execution.\']}\n    write_json(output / \'training_run.json\', record)\n    write_json(output / \'feature_order.json\', FEATURES)\n    for source, name in [(dataset, \'input_dataset.csv\'), (manifest_path, \'dataset_manifest.json\'),\n                         (reference_path, \'current_model_reference.json\'), (__file__, \'train_random_forest.py\')]:\n        (output / name).write_bytes(Path(source).read_bytes())\n    write_json(output / \'artifact_hashes.json\', {p.name: file_hash(p) for p in sorted(output.iterdir()) if p.is_file()})\n    archive = output.with_suffix(\'.zip\')\n    with zipfile.ZipFile(archive, \'w\', compression=zipfile.ZIP_DEFLATED) as zipped:\n        for path in sorted(output.iterdir()):\n            zipped.write(path, arcname=output.name + \'/\' + path.name)\n    print(\'Random Forest training finished. Keep the executed notebook with outputs.\', flush=True)\n    return output, archive, record\n\n\nif __name__ == \'__main__\':\n    import argparse\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--dataset\', type=Path, default=Path(\'seven_crop_without_npk.csv\'))\n    parser.add_argument(\'--manifest\', type=Path, default=Path(\'dataset_manifest.json\'))\n    parser.add_argument(\'--reference\', type=Path, default=Path(\'current_model_reference.json\'))\n    parser.add_argument(\'--output\', type=Path, default=Path(\'runs\'))\n    args = parser.parse_args()\n    run_training(args.dataset, args.manifest, args.reference, args.output)\n'
MANIFEST = json.loads("{\"schema_version\": 1, \"purpose\": \"existing_augmented_data_research_comparison_without_npk\", \"dataset_sha256\": \"bfd1f5b9c2a1b745b62f93f16e95464f3d2bb7e81f4bbaeb3621a986c134c3c1\", \"feature_order\": [\"temperature\", \"humidity\", \"ph\", \"rainfall\", \"ndvi\", \"elevation\", \"slope\"], \"classes\": [\"banana\", \"cacao\", \"coconut\", \"durian\", \"mango\", \"rice\", \"rubber\"], \"source_files\": {\"backend/Crop_recommendation_with_gis.csv\": \"1a6dcf3b22c20ff51c0ef7eee310df3ad5d359263d78d1a93e2dbaa7245ec03a\", \"backend/Crop_recommendation.csv\": \"54a5a6e5408668e668667efc50de2fc867c1b875e0431b4f54dd331b0a109a4e\"}, \"input_rows\": 4420, \"selected_rows\": 1820, \"other_crop_rows_excluded\": 2600, \"class_and_provenance_counts\": {\"banana\": {\"legacy_augmented_profile_row\": 160, \"matches_original_kaggle_feature_row\": 100}, \"cacao\": {\"legacy_augmented_profile_row\": 260, \"matches_original_kaggle_feature_row\": 0}, \"coconut\": {\"legacy_augmented_profile_row\": 160, \"matches_original_kaggle_feature_row\": 100}, \"durian\": {\"legacy_augmented_profile_row\": 260, \"matches_original_kaggle_feature_row\": 0}, \"mango\": {\"legacy_augmented_profile_row\": 160, \"matches_original_kaggle_feature_row\": 100}, \"rice\": {\"legacy_augmented_profile_row\": 160, \"matches_original_kaggle_feature_row\": 100}, \"rubber\": {\"legacy_augmented_profile_row\": 260, \"matches_original_kaggle_feature_row\": 0}}, \"provenance_labels\": {\"matches_original_kaggle_feature_row\": \"Exact match on the seven original non-GIS columns and target; not verified farm observations. GIS fields remain generated legacy features.\", \"legacy_augmented_profile_row\": \"Existing augmented crop-profile row; not a newly observed farm or a new record generated by this notebook.\"}, \"features\": {\"temperature\": {\"source\": \"Existing augmented CSV; unchanged predictor values\", \"period\": \"unresolved legacy source period\", \"method\": \"mixed original rows and generated crop profiles\"}, \"humidity\": {\"source\": \"Existing augmented CSV; unchanged predictor values\", \"period\": \"unresolved legacy source period\", \"method\": \"mixed original rows and generated crop profiles\"}, \"ph\": {\"source\": \"Existing augmented CSV; unchanged predictor values\", \"period\": \"legacy definition; not independently field verified\", \"method\": \"mixed original rows and generated crop profiles\"}, \"rainfall\": {\"source\": \"Existing augmented CSV; unchanged predictor values\", \"period\": \"unresolved legacy source period\", \"method\": \"mixed original rows and generated crop profiles\"}, \"ndvi\": {\"source\": \"Existing augmented CSV; unchanged predictor values\", \"period\": \"legacy definition; not independently field verified\", \"method\": \"generated crop-profile GIS proxy\"}, \"elevation\": {\"source\": \"Existing augmented CSV; unchanged predictor values\", \"period\": \"legacy definition; not independently field verified\", \"method\": \"generated crop-profile GIS proxy\"}, \"slope\": {\"source\": \"Existing augmented CSV; unchanged predictor values\", \"period\": \"legacy definition; not independently field verified\", \"method\": \"generated crop-profile GIS proxy\"}}, \"npk_included\": false, \"rows_generated\": 0, \"annualization_performed\": false, \"independent_farm_ids_available\": false, \"application_integration_ready\": false}")
CURRENT_REFERENCE = json.loads("{\"algorithm\": \"Random Forest\", \"estimator\": \"RandomForestClassifier\", \"crop_classes\": [\"banana\", \"cacao\", \"coconut\", \"durian\", \"mango\", \"rice\", \"rubber\"], \"feature_order\": [\"temperature\", \"humidity\", \"ph\", \"rainfall\", \"ndvi\", \"elevation\", \"slope\"], \"baseline_parameters\": {\"n_estimators\": 500, \"max_depth\": 14, \"min_samples_leaf\": 6, \"min_samples_split\": 12, \"max_features\": \"sqrt\", \"class_weight\": \"balanced_subsample\", \"random_state\": 42, \"n_jobs\": 1}, \"installed_artifact_sha256\": \"6cefb5545173c05521a5c1fe71e5cb654b2a17161a1e1399d88f93afa867c8cd\", \"dataset_sha256\": \"bfd1f5b9c2a1b745b62f93f16e95464f3d2bb7e81f4bbaeb3621a986c134c3c1\", \"original_run_id\": \"20261106T080700Z-3adeb48c\", \"original_training_timestamp_utc\": \"2026-11-06T08:07:12.723479+00:00\", \"npk_used\": false, \"scope\": \"Reference identity of the existing artifact, not results from this notebook.\"}")
DATASET_PATH = WORK / "seven_crop_without_npk.csv"
MANIFEST_PATH = WORK / "dataset_manifest.json"
REFERENCE_PATH = WORK / "current_model_reference.json"
TRAINER_PATH = WORK / "train_random_forest.py"
DATASET_PATH.write_bytes(base64.b64decode(DATASET_BASE64))
MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
REFERENCE_PATH.write_text(json.dumps(CURRENT_REFERENCE, indent=2), encoding="utf-8")
TRAINER_PATH.write_bytes(TRAINER_SOURCE.encode("utf-8"))
namespace = {"__name__": "rf_colab_trainer", "__file__": str(TRAINER_PATH.resolve())}
exec(compile(TRAINER_SOURCE, str(TRAINER_PATH), "exec"), namespace)


## 4. Inspect the preparation

These are dataset counts, not performance results. Exact duplicates are removed before the 80/20 stratified split. Five-fold cross-validation uses only training rows; the test set stays separate.


In [ ]:
import pandas as pd
from IPython.display import display, Image
frame, manifest, audit = namespace["prepare_data"](DATASET_PATH, MANIFEST_PATH)
print("Algorithm: Random Forest")
print("Inputs:", namespace["FEATURES"])
print("Crop classes:", namespace["LABELS"])
print("Dataset SHA-256:", audit["dataset_sha256"])
print("Source rows:", audit["input_rows"], "Unique rows:", audit["unique_rows"])
display(frame.head())
display(frame.groupby("label").size().rename("unique_rows").to_frame())


## 5. Results or actual training

With `RUN_TRAINING = False`, only the N/A table is shown. When enabled, this fits only Random Forest and computes results from this run. The model configuration follows the existing baseline; this notebook does not claim to compare or select algorithms.


In [ ]:
output_dir = results_zip = run = None
if RUN_TRAINING:
    output_dir, results_zip, run = namespace["run_training"](
        DATASET_PATH, MANIFEST_PATH, REFERENCE_PATH, WORK / "runs",
        seed=SEED, trees=FOREST_TREES, cv_folds=CV_FOLDS,
        execution_platform=PLATFORM, researcher=RESEARCHER_NAME.strip() or "Not supplied")
    display(pd.read_csv(output_dir / "metrics.csv"))
    display(pd.read_csv(output_dir / "classification_report.csv", index_col=0))
    display(Image(filename=str(output_dir / "confusion_matrix.png")))
else:
    display(pd.DataFrame([{ "Algorithm": "Random Forest", "Accuracy": "N/A",
        "Precision": "N/A", "Recall": "N/A", "F1-score": "N/A" }]))
    print("Training disabled. No evaluation was performed by this notebook.")


## 6. Save the evidence

For Chapter 2, save the preparation code and N/A table. After actual training, save the executed notebook **with outputs** and the downloaded ZIP. That ZIP includes the metrics, confusion matrix, fold scores, train/test assignments, predictions, model, settings and hashes. Run metadata supports replay but does not independently prove where execution occurred.


In [ ]:
if results_zip is not None:
    print("Fresh run:", run["run_id"])
    print("Evidence:", results_zip)
    if IN_COLAB:
        from google.colab import files
        files.download(str(results_zip))
else:
    print("No completed training evidence yet; results remain N/A.")


## Data limits

The unchanged CSV mixes original Kaggle-based rows with generated crop profiles and GIS predictors. Cacao, durian and rubber use generated profiles. Independent farm grouping and consistent labelled climate periods are unavailable. Any computed scores describe this dataset experiment, not verified field or annual-input accuracy. Banana is one crop class; these labels do not distinguish varieties. A newly trained artifact does not automatically replace the installed application model.
